# 🧩 Notebook 2: UML → Python

## 🛠️ Setup

```bash
cd 07-object-oriented-design/uml-basics
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In this notebook we turn a UML sketch into real Python code, so you can see the mapping
from diagram → class → relationship.

We'll model the Library from notebook 1.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from abc import ABC, abstractmethod


# ─────────────────────────────────────────────
# Author ──(1..*)── Book   (association, 1 author per book)
# ─────────────────────────────────────────────
@dataclass
class Author:
    name: str


# ─────────────────────────────────────────────
# Book (abstract base) △ PrintBook, Ebook, Audiobook   (inheritance)
# ─────────────────────────────────────────────
class Book(ABC):
    def __init__(self, title: str, author: Author):
        self.title = title
        self.author = author

    @abstractmethod
    def media_type(self) -> str: ...

    def __repr__(self):
        return f"{self.media_type()}({self.title!r} by {self.author.name})"


class PrintBook(Book):
    def media_type(self): return "PrintBook"

class Ebook(Book):
    def media_type(self): return "Ebook"

class Audiobook(Book):
    def media_type(self): return "Audiobook"


In [ ]:
# ─────────────────────────────────────────────
# Library ──◆── Book  (composition: Library owns its Books)
# ─────────────────────────────────────────────
@dataclass
class Library:
    name: str
    books: list[Book] = field(default_factory=list)

    def add(self, book: Book) -> None:
        self.books.append(book)

    def by_author(self, author_name: str) -> list[Book]:
        return [b for b in self.books if b.author.name == author_name]


tolkien = Author("J.R.R. Tolkien")
rowling = Author("J.K. Rowling")

lib = Library("Central")
lib.add(PrintBook("The Hobbit", tolkien))
lib.add(Ebook("LOTR", tolkien))
lib.add(Audiobook("Harry Potter 1", rowling))

for b in lib.books:
    print(b)

print("Tolkien books:", lib.by_author("J.R.R. Tolkien"))


## Simulating a sequence diagram

The earlier sequence diagram becomes a chain of method calls. Watching the print output
is essentially the same thing as reading the sequence diagram top-to-bottom.

In [ ]:
class PaymentSvc:
    def charge(self, user, amount):
        print(f"  PaymentSvc.charge({user}, ${amount}) → ok")
        return "ok"

class OrderSvc:
    def __init__(self, payments: PaymentSvc):
        self.payments = payments
        self._next_id = 41

    def create_order(self, user, amount):
        print(f" OrderSvc.create_order({user})")
        self.payments.charge(user, amount)
        self._next_id += 1
        return self._next_id

class WebApp:
    def __init__(self, orders: OrderSvc):
        self.orders = orders

    def checkout(self, user, amount):
        print(f"WebApp.checkout({user}) — user clicked 'Pay'")
        order_id = self.orders.create_order(user, amount)
        print(f"WebApp returns 200 OK, order #{order_id}")
        return order_id

app = WebApp(OrderSvc(PaymentSvc()))
app.checkout("alice", 42)


## Try it

1. Add a `Loan` class that associates a `Book` with a `User` (association, not composition — because users outlive loans).
2. Draw the updated class diagram in ASCII in a markdown cell.
3. Write a sequence diagram for `user.borrow(book)` showing calls to `Library`, `Loan`, `NotificationSvc`.